### Python and Pandas

In [1]:
import pandas as pd

In [13]:
file_path = "pm_portfolio_reports.xlsx"


# Read all sheets dynamically
all_sheets = pd.read_excel(file_path, sheet_name=None)

dfs = []

for sheet_name, df in all_sheets.items():
    # Skip non-project sheets
    if sheet_name.lower() == "portfolio":
        continue
    
    # 1. Remove empty columns (Unnamed...)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # 2. Strip column names
    df.columns = df.columns.str.strip()

    # 3. Remove fully empty rows
    df = df.dropna(how="all")

    # 4. Add month info
    df["month"] = sheet_name

    dfs.append(df)

# Concatenate everything
df_projects = pd.concat(dfs, ignore_index=True)


In [ ]:
# Head method: It prints the first n rows of the DataFrame. By default, it shows the first 5 rows, but we can specify a different number by passing an argument (e.g., df.head(10) to show the first 10 rows).
df_projects.head(15)

,Project_ID,Project_Name,Portfolio_ID,PM,Planned_Budget,Actual_Budget,Planned_Progress (%),Actual_Progress (%),month
0,P1,Billing,PF1,Nikola,100000,95000,30,25,2026_01
1,P2,AI Engine,PF1,Elena,200000,210000,40,35,2026_01
2,P3,Mobile App,PF2,Stefan,150000,140000,50,55,2026_01
3,P1,Billing,PF1,Nikola,100000,110000,60,55,2026_02
4,P2,AI Engine,PF1,Elena,200000,230000,70,60,2026_02
5,P3,Mobile App,PF2,Stefan,150000,155000,80,75,2026_02
6,P1,Billing,PF1,Nikola,100000,120000,100,90,2026_03
7,P2,AI Engine,PF1,Elena,200000,250000,100,85,2026_03
8,P3,Mobile App,PF2,Stefan,150000,160000,100,100,2026_03


In [ ]:
# Display data types to check for any inconsistencies
# dtypes method: It returns the data types of each column in the DataFrame. This is useful for understanding the structure of the data and for identifying any potential issues with data types (e.g., numeric columns being read as objects due to non-numeric values).
df_projects.dtypes

Project_ID              object
Project_Name            object
Portfolio_ID            object
PM                      object
Planned_Budget           int64
Actual_Budget            int64
Planned_Progress (%)     int64
Actual_Progress (%)      int64
month                   object
dtype: object

### <span style="color: #2E86AB;">Q: Give an example of a calculated/computed column in a database table.</span>

### 1. Budget Overrun

In [ ]:
# Calculate budget variance as a new column (caculated column) and add it to the DataFrame
df_projects["budget_variance"] = df_projects["Actual_Budget"] - df_projects["Planned_Budget"]

In [ ]:
# Dummy column for showcasing how to add a new column with a default value (a constant value for all rows)
df_projects["test_1"] = 1

In [18]:
df_projects.head()

,Project_ID,Project_Name,Portfolio_ID,PM,Planned_Budget,Actual_Budget,Planned_Progress (%),Actual_Progress (%),month,budget_variance,test_1
0,P1,Billing,PF1,Nikola,100000,95000,30,25,2026_01,-5000,1
1,P2,AI Engine,PF1,Elena,200000,210000,40,35,2026_01,10000,1
2,P3,Mobile App,PF2,Stefan,150000,140000,50,55,2026_01,-10000,1
3,P1,Billing,PF1,Nikola,100000,110000,60,55,2026_02,10000,1
4,P2,AI Engine,PF1,Elena,200000,230000,70,60,2026_02,30000,1


### 2. Schedule Delay

In [19]:
df_projects["progress_variance"] = df_projects["Actual_Progress (%)"] - df_projects["Planned_Progress (%)"]

In [20]:
df_projects.head()

,Project_ID,Project_Name,Portfolio_ID,PM,Planned_Budget,Actual_Budget,Planned_Progress (%),Actual_Progress (%),month,budget_variance,test_1,progress_variance
0,P1,Billing,PF1,Nikola,100000,95000,30,25,2026_01,-5000,1,-5
1,P2,AI Engine,PF1,Elena,200000,210000,40,35,2026_01,10000,1,-5
2,P3,Mobile App,PF2,Stefan,150000,140000,50,55,2026_01,-10000,1,5
3,P1,Billing,PF1,Nikola,100000,110000,60,55,2026_02,10000,1,-5
4,P2,AI Engine,PF1,Elena,200000,230000,70,60,2026_02,30000,1,-10


### <span style="color: #2E86AB;">Q: What window functions do we know from the previous module? Let's name 3 of them!</span>

### 3. Track progress change month-over-month

In [ ]:
# To calculate the progress delta, we need to sort the DataFrame by Project_ID and month to ensure that we are comparing the correct rows for each project. Then, we can use the shift() method to get the previous month's progress for each project and calculate the difference.

# Please focus on the .shift() method, which is used to access the previous row's value within each project group. This allows us to calculate the month-over-month change in progress for each project.

df_projects = df_projects.sort_values(["Project_ID", "month"])

df_projects["prev_progress"] = df_projects.groupby("Project_ID")["Actual_Progress (%)"].shift(1)

df_projects["progress_delta"] = (
    df_projects["Actual_Progress (%)"] - df_projects["prev_progress"]
)
df_projects.head(15)

,Project_ID,Project_Name,Portfolio_ID,PM,Planned_Budget,Actual_Budget,Planned_Progress (%),Actual_Progress (%),month,budget_variance,test_1,progress_variance,prev_progress,progress_delta
0,P1,Billing,PF1,Nikola,100000,95000,30,25,2026_01,-5000,1,-5,NaN,NaN
3,P1,Billing,PF1,Nikola,100000,110000,60,55,2026_02,10000,1,-5,25.0,30.0
6,P1,Billing,PF1,Nikola,100000,120000,100,90,2026_03,20000,1,-10,55.0,35.0
1,P2,AI Engine,PF1,Elena,200000,210000,40,35,2026_01,10000,1,-5,NaN,NaN
4,P2,AI Engine,PF1,Elena,200000,230000,70,60,2026_02,30000,1,-10,35.0,25.0
7,P2,AI Engine,PF1,Elena,200000,250000,100,85,2026_03,50000,1,-15,60.0,25.0
2,P3,Mobile App,PF2,Stefan,150000,140000,50,55,2026_01,-10000,1,5,NaN,NaN
5,P3,Mobile App,PF2,Stefan,150000,155000,80,75,2026_02,5000,1,-5,55.0,20.0
8,P3,Mobile App,PF2,Stefan,150000,160000,100,100,2026_03,10000,1,0,75.0,25.0


### 4. Which portfolio is underperforming relative to its budget?

In [9]:
# Join with portfolio 
df_portfolio = all_sheets["portfolio"]

df = df_projects.merge(df_portfolio, on="Portfolio_ID", how="left")

In [10]:
df.head(15)

,Project_ID,Project_Name,Portfolio_ID,PM,Planned_Budget,Actual_Budget,Planned_Progress (%),Actual_Progress (%),month,budget_variance,progress_variance,prev_progress,progress_delta,Portfolio_Name,Director,Strategic_Goal,Total_Budget
0,P1,Billing,PF1,Nikola,100000,95000,30,25,2026_01,-5000,-5,NaN,NaN,Digital Transformation,Marin,Modernize core systems,500000
1,P1,Billing,PF1,Nikola,100000,110000,60,55,2026_02,10000,-5,25.0,30.0,Digital Transformation,Marin,Modernize core systems,500000
2,P1,Billing,PF1,Nikola,100000,120000,100,90,2026_03,20000,-10,55.0,35.0,Digital Transformation,Marin,Modernize core systems,500000
3,P2,AI Engine,PF1,Elena,200000,210000,40,35,2026_01,10000,-5,NaN,NaN,Digital Transformation,Marin,Modernize core systems,500000
4,P2,AI Engine,PF1,Elena,200000,230000,70,60,2026_02,30000,-10,35.0,25.0,Digital Transformation,Marin,Modernize core systems,500000
5,P2,AI Engine,PF1,Elena,200000,250000,100,85,2026_03,50000,-15,60.0,25.0,Digital Transformation,Marin,Modernize core systems,500000
6,P3,Mobile App,PF2,Stefan,150000,140000,50,55,2026_01,-10000,5,NaN,NaN,Customer Experience,Ivana,Improve mobile and UX,300000
7,P3,Mobile App,PF2,Stefan,150000,155000,80,75,2026_02,5000,-5,55.0,20.0,Customer Experience,Ivana,Improve mobile and UX,300000
8,P3,Mobile App,PF2,Stefan,150000,160000,100,100,2026_03,10000,0,75.0,25.0,Customer Experience,Ivana,Improve mobile and UX,300000


In [11]:
# Aggregate at portfolio level

portfolio_perf = df.groupby("Portfolio_Name").agg({
    "budget_variance": "sum",
    "progress_variance": "sum",
    "Actual_Progress (%)": "mean",
    "Actual_Budget": "sum",
    "Total_Budget": "max"
}).reset_index()

portfolio_perf.head()

,Portfolio_Name,budget_variance,progress_variance,Actual_Progress (%),Actual_Budget,Total_Budget
0,Customer Experience,5000,0,76.666667,455000,300000
1,Digital Transformation,115000,-50,58.333333,1015000,500000
